<a href="https://colab.research.google.com/github/missstechie/Online-Internship-I-HUB-Data-IIITH-Vision-Tasks-using-Generative-AI/blob/main/Indian_Legal_Q_A_using_LoRA_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Indian Legal Question Answering using LoRA Fine-Tuning
#Objective

Fine-tune the Qwen2.5-7B language model using Low-Rank Adaptation (LoRA) on an Indian legal instruction-response dataset to develop a domain-specific Indian Legal Question Answering model.

#Domain

Indian Legal Question Answering

#Project Overview

This project explores domain-specific fine-tuning of a large language model using LoRA/QLoRA techniques. The Qwen2.5-7B model is loaded in 4-bit precision and adapted using LoRA on an Indian legal instruction-response dataset.

#Project Pipeline

Qwen2.5-7B

↓

4-bit Quantization (QLoRA)

↓

LoRA Adapters

↓

Indian Legal Dataset

↓

Supervised Fine-Tuning

↓

Indian Legal Q&A Model


#Expected Features

Indian legal domain specialization

Legal question answering

Improved understanding of Indian legal terminology

Instruction-response based generation

Parameter-efficient LoRA fine-tuning

Lightweight model adaptation


#Important Disclaimer

This model is developed for educational and research purposes. Its responses should not be considered professional legal advice.

#Project Information

Component -	Details

Project - Indian Legal Question Answering

Base Model -	Qwen2.5-7B

Fine-Tuning Method -	LoRA

Quantization -	4-bit / QLoRA

Dataset	 - Indian Legal Data v2

Initial Dataset Size	- 5,000 examples

Original Dataset Size	- 171,640 examples

Hardware -	Google Colab Tesla T4 GPU

Framework	- PyTorch

Fine-Tuning Library	- Unsloth

Dataset Library -	Hugging Face Datasets

Training Framework -	TRL

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


GPU available: True
GPU: Tesla T4


#Environment Setup

Unsloth is used to perform memory-efficient and parameter-efficient fine-tuning of the Qwen2.5-7B model using LoRA/QLoRA.

In [1]:
!pip install --upgrade --no-cache-dir unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.6/72.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 MB 317.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 247.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 379.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 262.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 324.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 410.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 355.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 375.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 389.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 446.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 172.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Environment Verification

Before loading the base model, we verify that the required libraries and the Tesla T4 GPU are available.

In [1]:
import torch
import transformers
import trl
import unsloth

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("Unsloth:", unsloth.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1527: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
Transformers: 5.5.0
TRL: 0.24.0
Unsloth: 2026.8.10
GPU available: True
GPU: Tesla T4


#Load the Base Model

The base model used in this project is Qwen2.5-7B. The model is loaded in 4-bit precision to reduce GPU memory usage and make fine-tuning possible on a Tesla T4 GPU.

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


==((====))==  Unsloth 2026.8.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
#%%capture
#!pip install unsloth
# Also get the latest nightly Unsloth!
#!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
#from unsloth import FastLanguageModel
#import torch
#max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
#dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
#load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
#fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

#model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    #model_name = "unsloth/Qwen2.5-7B",
    #max_seq_length = max_seq_length,
    #dtype = dtype,
    #load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.9.post1: Fast Qwen2 patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.4.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.87k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

#Apply LoRA Adapters

Low-Rank Adaptation (LoRA) is used to fine-tune the Qwen2.5-7B model without updating all of its parameters. Instead, trainable low-rank adapter layers are added to selected model modules.

This reduces memory requirements and makes parameter-efficient fine-tuning possible on the available Tesla T4 GPU.

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.8.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


#Indian Legal Dataset

For domain-specific fine-tuning, the generic Alpaca dataset used in the original template is replaced with the Indian Legal Data v2 dataset.

The dataset contains instruction-response pairs related to Indian law. Each example consists of an instruction, representing the legal question or task, and a response, representing the corresponding legal answer.

The complete dataset contains 171,640 examples. For the initial experiment, a smaller subset will be used to make training practical on the available Tesla T4 GPU.

In [6]:
from datasets import load_dataset

# Load Indian Legal Dataset
dataset = load_dataset(
    "kaushik-harsh-99/Indian-legal-data-v2",
    split="train"
)

print(dataset)


README.md:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B /  565MB            

train.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/171640 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 171640
})


In [ ]:
#alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

#Dataset Sampling

The complete Indian Legal Data v2 dataset contains 171,640 instruction-response examples. For the initial experiment, 5,000 examples are selected.

The dataset is shuffled using a fixed random seed to ensure that the selected subset is reproducible.

In [7]:
# Shuffle the dataset
dataset = dataset.shuffle(seed=3407)

# Select 5,000 examples for the initial experiment
dataset = dataset.select(range(5000))

print("Number of training examples:", len(dataset))


Number of training examples: 5000


#Dataset Formatting

The Indian Legal Data v2 dataset contains two relevant fields: instruction and response.

These fields are converted into a single formatted text field containing the legal instruction and its corresponding response. This format can then be passed to the supervised fine-tuning trainer.

In [8]:
alpaca_prompt = """Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    responses = examples["response"]

    texts = []

    for instruction, response in zip(instructions, responses):
        text = alpaca_prompt.format(
            instruction,
            response
        ) + EOS_TOKEN

        texts.append(text)

    return {"text": texts}


In [9]:
dataset = dataset.map(
    formatting_prompts_func,
    batched=True
)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [10]:
print(dataset[0]["text"])


Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
Explain the provisions of Section 17 of the Mental Health Act, 1987 concerning the admission and regulation of voluntary patients.

### Response:
Heading: Section 17: Admission of, and regulation with respect to, voluntary patients

1. Introduction:
• Section 17 of the Mental Health Act, 1987, deals specifically with the procedure for admitting individuals who seek treatment as voluntary patients in psychiatric hospitals or nursing homes, and establishes the regulatory framework that these patients must follow.
• It outlines the process by which an applicant can be admitted and defines the subsequent obligations placed upon the admitted patient.

2. Text Breakdown:
• Subsection (1) details the procedure for admission. It mandates that upon receiving a request (under Section 15 or Section 16), the medical officer in charge must conduct an inquiry within a maximum period of tw

#Supervised Fine-Tuning

The Qwen2.5-7B model with LoRA adapters is fine-tuned using the Supervised Fine-Tuning (SFT) method provided by Hugging Face TRL.

For the initial experiment, training is limited to 60 steps. This short run is used to verify that the complete training pipeline works correctly on the Tesla T4 GPU before performing a longer experiment.

In [12]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="outputs",

    # Training
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,

    # Logging
    logging_steps=1,

    # Precision
    fp16=True,
    bf16=False,

    # Dataset
    dataset_text_field="text",
    max_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,

    # Optimizer
    optim="adamw_8bit",

    # Reproducibility
    seed=3407,

    # Reporting
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
#from trl import SFTTrainer
#from transformers import TrainingArguments
#from unsloth import is_bfloat16_supported

#trainer = SFTTrainer(
    #model = model,
    #tokenizer = tokenizer,
    #train_dataset = dataset,
    #dataset_text_field = "text",
    #dataset_num_proc = 2,
    #packing = False, # Can make training 5x faster for short sequences.
    #args = TrainingArguments(
        #per_device_train_batch_size = 2,
        #gradient_accumulation_steps = 4,
        #warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        #max_steps = 60,
        #learning_rate = 2e-4,
        #fp16 = not is_bfloat16_supported(),
        #bf16 = is_bfloat16_supported(),
        #optim = "adamw_8bit",
        #weight_decay = 0.01,
        #lr_scheduler_type = "linear",
        #seed = 3407,
        #output_dir = "outputs",
        #report_to = "none", # Use this for WandB etc
    #),
#)

Map (num_proc=2):   0%|          | 0/51760 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


#Model Fine-Tuning

The LoRA-adapted Qwen2.5-7B model is now fine-tuned on the selected 5,000 Indian legal instruction-response examples.

A 60-step training run is used for the initial experiment to verify that the training pipeline works successfully on the Tesla T4 GPU.

In [13]:
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.269807
2,1.357750
3,1.156546
4,1.341191
5,1.148043
6,1.350357
7,1.078216
8,1.150806
9,1.077554
10,1.102898


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
#@title Show current memory stats
#gpu_stats = torch.cuda.get_device_properties(0)
#start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
#max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
##print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.748 GB.
5.764 GB of memory reserved.


In [ ]:
#trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 40,370,176


Step,Training Loss
1,0.984700
2,1.079000
3,1.063000
4,1.096100
5,1.024800
6,0.967800
7,0.776300
8,1.000400
9,0.891900
10,0.946700


#Training Results

The Qwen2.5-7B model was fine-tuned using LoRA on 5,000 examples from the Indian Legal Data v2 dataset. The initial experiment used 60 training steps on a Google Colab Tesla T4 GPU.

In [14]:
# Display training statistics
print(trainer_stats)


TrainOutput(global_step=60, training_loss=1.0021188348531722, metrics={'train_runtime': 1067.0726, 'train_samples_per_second': 0.45, 'train_steps_per_second': 0.056, 'total_flos': 1.3234375984392192e+16, 'train_loss': 1.0021188348531722, 'epoch': 0.096})


In [15]:
used_memory = round(
    torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024,
    3
)

print(f"Peak reserved GPU memory = {used_memory} GB")


Peak reserved GPU memory = 10.344 GB


In [ ]:
#@title Show final memory and time stats
#used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
#used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
#used_percentage = round(used_memory         /max_memory*100, 3)
#lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
#print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
#print(f"Peak reserved memory = {used_memory} GB.")
#print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
#print(f"Peak reserved memory % of max memory = {used_percentage} %.")
#print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

462.3942 seconds used for training.
7.71 minutes used for training.
Peak reserved memory = 7.893 GB.
Peak reserved memory for training = 2.129 GB.
Peak reserved memory % of max memory = 53.519 %.
Peak reserved memory for training % of max memory = 14.436 %.


#Legal Question Answering — Model Testing

After fine-tuning, the LoRA-adapted Qwen2.5-7B model is evaluated using questions from the Indian legal domain.

The objective is to observe whether the fine-tuned model can generate relevant responses to Indian legal questions.

In [16]:
FastLanguageModel.for_inference(model)

question = "What is Article 32 of the Indian Constitution?"

inputs = tokenizer(
    [
        alpaca_prompt.format(
            question,
            ""
        )
    ],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True
)

print(
    tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]
)


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is Article 32 of the Indian Constitution?

### Response:
Article 32 of the Indian Constitution is a fundamental provision that guarantees the right to move the Supreme Court for the enforcement of any of the fundamental rights guaranteed by the Constitution. This article is a cornerstone of the Indian legal system, ensuring that citizens have a direct and effective means to seek redress for violations of their fundamental rights. The article provides a mechanism for individuals to challenge the actions of the executive and legislative branches of government, thereby safeguarding the integrity of the Constitution and the rule of law. The article is a critical component of the Indian legal framework, ensuring that the fundamental rights of citizens are protected and enforced. The article is a fundamental provision of the Indian Constitution, ensuring that citizens have a 

In [17]:
questions = [
    "What is Article 32 of the Indian Constitution?",
    "What is judicial review in India?",
    "What is the purpose of a writ petition?",
    "What is the difference between a civil case and a criminal case?",
    "What is Article 21 of the Indian Constitution?"
]

FastLanguageModel.for_inference(model)

for i, question in enumerate(questions, 1):

    inputs = tokenizer(
        [
            alpaca_prompt.format(
                question,
                ""
            )
        ],
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
    )

    answer = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    print("=" * 100)
    print(f"TEST {i}")
    print(f"QUESTION: {question}")
    print()
    print(answer)
    print()


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 1
QUESTION: What is Article 32 of the Indian Constitution?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is Article 32 of the Indian Constitution?

### Response:
Article 32 of the Indian Constitution is a fundamental provision that guarantees the right to move the Supreme Court for the enforcement of any of the fundamental rights guaranteed by the Constitution. This article is a cornerstone of the Indian legal system, ensuring that citizens have a direct and effective means to seek redress for violations of their fundamental rights. The article provides a mechanism for individuals to challenge the actions of the executive and legislative branches of government, thereby safeguarding the integrity of the Constitution and the rule of law. The article is a critical component of the Indian legal framework, ensuring that the fundamental rights of citizens are protected and enforced. The article is a fundamental pro

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 2
QUESTION: What is judicial review in India?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is judicial review in India?

### Response:
## Introduction
Judicial review is a fundamental aspect of the Indian legal system, serving as a critical mechanism for ensuring the proper functioning of the judiciary and the protection of individual rights. It allows the judiciary to scrutinize and challenge the actions of the executive and legislative branches, ensuring that they adhere to the principles of the Constitution and the rule of law. This process is essential for maintaining the integrity of the legal system and upholding the rights of citizens.

### Subheading 1: The Role of Judicial Review
Judicial review is a process by which the judiciary can review and challenge the actions of the executive and legislative branches. It is a mechanism for ensuring that the actions of these branches are in line with the prin

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 3
QUESTION: What is the purpose of a writ petition?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is the purpose of a writ petition?

### Response:
A writ petition is a legal document that serves as a formal request to a higher court or tribunal for a specific remedy or relief. It is typically filed when an individual or entity believes that their rights have been violated or that a legal injustice has occurred. The purpose of a writ petition is to seek redress for the wrong suffered, to challenge the legality of a decision or action, or to obtain a specific order or injunction. Writ petitions are often used in cases involving constitutional rights, administrative actions, or disputes over property or contracts. The process of filing a writ petition involves presenting the case to the court, which then reviews the evidence and arguments to determine whether the writ should be granted. If the writ is granted, 

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 4
QUESTION: What is the difference between a civil case and a criminal case?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is the difference between a civil case and a criminal case?

### Response:
## Introduction
In the realm of law, the distinction between civil and criminal cases is crucial. While both types of cases involve legal disputes, they differ significantly in their nature, purpose, and the legal processes involved. Understanding these differences is essential for anyone seeking to navigate the legal system effectively.

### Civil Cases
Civil cases are primarily concerned with disputes between individuals or entities over property, contracts, or other civil matters. These cases are typically initiated by one party seeking to enforce a right or remedy against another. The primary goal of a civil case is to resolve the dispute and provide a fair and just outcome for the parties involved. The legal p

#Save the Fine-Tuned LoRA Model

The trained LoRA adapter and tokenizer are saved locally so that the fine-tuned model can be reused for inference and evaluation without retraining.

In [18]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("LoRA model saved successfully.")


Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


LoRA model saved successfully.


#Before vs After Fine-Tuning

To evaluate the effect of domain-specific fine-tuning, the same legal questions will be tested using the original Qwen2.5-7B model and the LoRA-fine-tuned model.

The base-model responses will be compared with the responses generated after LoRA fine-tuning.

In [19]:
import gc
import torch

del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")


GPU memory cleared.


In [1]:
from unsloth import FastLanguageModel

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("Base Qwen2.5-7B loaded successfully.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base Qwen2.5-7B loaded successfully.


In [4]:
alpaca_prompt = """Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
{}

### Response:
{}"""

base_questions = [
    "What is Article 32 of the Indian Constitution?",
    "What is judicial review in India?",
    "What is the purpose of a writ petition?",
    "What is the difference between a civil case and a criminal case?",
    "What is Article 21 of the Indian Constitution?"
]

FastLanguageModel.for_inference(base_model)

for i, question in enumerate(base_questions, 1):

    inputs = base_tokenizer(
        [
            alpaca_prompt.format(
                question,
                ""
            )
        ],
        return_tensors="pt"
    ).to("cuda")

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
    )

    answer = base_tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    print("=" * 100)
    print(f"BASE MODEL TEST {i}")
    print(f"QUESTION: {question}")
    print()
    print(answer)
    print()


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL TEST 1
QUESTION: What is Article 32 of the Indian Constitution?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is Article 32 of the Indian Constitution?

### Response:
Article 32 of the Indian Constitution is a fundamental right that guarantees the right to move the Supreme Court for the enforcement of any of the fundamental rights granted under the Constitution. This article provides a mechanism for individuals to seek redressal against violations of their fundamental rights by the state or any other authority.

Here are some key points about Article 32:

1. **Right to Move Supreme Court**: Any person whose fundamental rights have been violated can approach the Supreme Court of India to seek enforcement of those rights. This includes both citizens and non-citizens who are within the territory of India.

2. **Nature of Relief**: The Supreme Court has the power to issue various forms of relief, including 

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL TEST 2
QUESTION: What is judicial review in India?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is judicial review in India?

### Response:
Judicial review in India refers to the power of the Supreme Court and High Courts to examine and determine the constitutionality of laws, executive actions, and administrative decisions. This concept is derived from the principle of judicial review, which is a fundamental aspect of the Indian Constitution and the legal system.

#### Key Points:
1. **Supreme Court and High Courts**: The Supreme Court of India and the High Courts have the authority to review and declare the validity of any law or action taken by the executive or legislative branches of government.
2. **Constitutional Validity**: Judicial review allows these courts to assess whether a law or action is in accordance with the provisions of the Indian Constitution. If a law or action is found to be uncon

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL TEST 3
QUESTION: What is the purpose of a writ petition?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is the purpose of a writ petition?

### Response:
A writ petition is a legal document filed in a higher court by an individual or entity seeking relief from a lower court's decision or order. The purpose of a writ petition is to challenge the legality of a decision or order made by a lower court, and to request that the higher court review and correct any errors or violations of law that may have occurred.

Writ petitions are typically used in cases where the lower court's decision or order is final and cannot be appealed, or where the lower court lacks jurisdiction to hear the case. They are also used in cases where the lower court's decision or order is based on a violation of constitutional or statutory rights.

The writ petition must be filed within a certain time frame, and must include specific i

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL TEST 4
QUESTION: What is the difference between a civil case and a criminal case?

Below is an instruction related to Indian law.
Provide a clear and informative legal response.

### Instruction:
What is the difference between a civil case and a criminal case?

### Response:
In India, civil cases and criminal cases are two distinct types of legal proceedings that serve different purposes and have different outcomes. Here's a clear and informative explanation of the key differences between them:

#### Civil Cases:
1. **Nature of Dispute**: Civil cases involve disputes between individuals or entities over rights, obligations, or damages. They typically arise from contractual agreements, property disputes, family matters (such as divorce), or personal injury claims.
2. **Purpose**: The primary purpose of a civil case is to resolve disputes and provide relief to the aggrieved party. This can include monetary compensation, specific performance of a contract, or an injunction agai